# Workbook 4 — "Inductive bias" is a four-letter word

**Solutions · Mysteries of Deep Learning, Iliad Intensive 2026**

"Neural networks generalize because of their inductive bias" is true, and as an
explanation it is empty: every learner that generalizes has an inductive bias. This
workbook works through the textbook chain from Occam's razor to Kolmogorov complexity
to Solomonoff's universal prior, then points to current accounts of what deep networks
prefer: short programs, small circuits, small norms, and structure in the data itself.

Route (sections are largely independent; the math is in §2–3 and in the exercises of
§4–5):

- [1. Inductive bias is a curiosity stopper](#wb4-section-1). Two sentences of orientation.
- [2. Occam, two ways](#wb4-section-2). Bayesian evidence (prior volume), Kolmogorov complexity (description length) and MDL two-part codes, and where they part. Derivation exercises: the Occam factor of a Gaussian model, the four basic facts about $K$, why MDL states parameters to precision $1/\sqrt n$, and two on where the notions disagree.
- [3. Solomonoff](#wb4-section-3). The universal prior and its $K(\mu)\ln 2$ regret bound, and Legg's limits on computable predictors, both derived as exercises; a pointer to simplicity bias in random networks. Core reading: Rathmanner and Hutter.
- [4. Program synthesis and the circuit prior](#wb4-section-4). Two readings (Furman's program-synthesis hypothesis; Hänni and Vaintrob's circuit-prior theorem) and one exercise: sparse parity for a counting learner versus a kernel learner.
- [5. Hierarchical representations](#wb4-section-5). The Random Hierarchy Model, with one estimation exercise.
- [6. The accounts side by side](#wb4-section-6). One table of the priors over functions, and where singular learning theory sits.

The notebook ends with links to related workbooks and further reading.

**How to use this notebook.** Pick what interests you; nobody is expected to do
everything. The exercises are mostly derivations with hints. Worked solutions are in
the solutions version of this notebook, which everyone may use. Each reading states its
result first so you can decide whether to open it; *core* marks the one to read if
you read only one in that section.

<a id="wb4-section-1"></a>

## 1. Inductive bias is a curiosity stopper

By no-free-lunch (WB2 §2), a learner that does well on the tasks we meet must favour
some targets over others; "inductive bias" names that preference without saying what
it is. §2–5 give candidate answers: prior volume and description length (§2–3),
program length, circuit size and kernel norm (§4), and structure in the data (§5).

<a id="wb4-section-2"></a>

## 2. Occam, two ways

**The result.** Bayesian model comparison penalizes a model by how much prior volume
it wastes on data that did not occur: the Occam factor. Kolmogorov complexity
penalizes a hypothesis by the length of its shortest description on a fixed universal
machine. MDL's two-part code sits between them. Evidence and two-part code agree at
leading order for regular parametric models and come apart in instructive cases, so
"simplicity" is not one thing.

### 2.A. Bayesian Occam: volume

[Source: MacKay, *Bayesian Interpolation* (1992).](https://authors.library.caltech.edu/records/r7qgh-q6g10)

A model $\mathcal M$ has parameters $\theta\in\mathbb R^p$, a prior $\pi(\theta)$
and a likelihood $p(D\mid\theta)$. Its *evidence* (marginal likelihood) is

$$
Z(\mathcal M)=\int p(D\mid\theta)\,\pi(\theta)\,d\theta,
$$

and two models are compared by $Z_1/Z_2$ times the prior odds.

With one parameter, a prior of width $\Delta_\pi$ and a likelihood sharply peaked at
$\hat\theta$ with width $\Delta_{\text{post}}$,

$$
Z\approx p(D\mid\hat\theta)\cdot\frac{\Delta_{\text{post}}}{\Delta_\pi}.
$$

The second factor, at most 1, is the **Occam factor**: the fraction of the prior's
bets that turned out compatible with the data. A flexible model spreads its prior
over many datasets it could have fit, so it put little probability on whichever one
arrived. Nothing here refers to how long the model is to write down.

With $p$ parameters and $n$ data points, the Laplace approximation gives
$-\ln Z=-\ln p(D\mid\hat\theta)+\tfrac p2\ln n+O(1)$, which is BIC (WB2 §7). For
singular models (the SLT day), $\tfrac p2$ is replaced at leading order by the
learning coefficient $\lambda\le\tfrac p2$. Either way the penalty measures the
*volume* of near-optimal parameters, not the parameter count.

#### 2.A.i. The Occam factor, exactly

Data $y_1,\dots,y_n$ are i.i.d. $\mathcal N(\theta,\sigma^2)$ with $\sigma$ known,
and the prior is $\theta\sim\mathcal N(0,\tau^2)$. Write $\bar y$ for the sample mean
and $s^2=\sigma^2/n$.

1. Show that $\sum_i(y_i-\theta)^2=\sum_i(y_i-\bar y)^2+n(\theta-\bar y)^2$, and hence
   $p(D\mid\theta)=p(D\mid\bar y)\,e^{-(\theta-\bar y)^2/2s^2}$, so $\hat\theta=\bar y$.
2. Show that
   $Z=p(D\mid\hat\theta)\cdot\sqrt{\dfrac{s^2}{\tau^2+s^2}}\,\exp\!\Big(-\dfrac{\bar y^2}{2(\tau^2+s^2)}\Big)$.
3. Show that for large $n$,
   $-\ln Z=-\ln p(D\mid\hat\theta)+\tfrac12\ln n+\tfrac12\ln\dfrac{\tau^2}{\sigma^2}+\dfrac{\bar y^2}{2\tau^2}+o(1)$,
   and match the terms with $\Delta_{\text{post}}/\Delta_\pi$.
4. Fix the data and let $\tau\to\infty$, an ever vaguer prior. What happens to $Z$,
   and why does this not mean the model fits worse?

<details><summary>Hint</summary>

For step 2, write $e^{-(\theta-\bar y)^2/2s^2}=\sqrt{2\pi s^2}\;\mathcal N(\bar y;\theta,s^2)$.
Integrating the density $\mathcal N(\theta;0,\tau^2)$ against
$\mathcal N(\bar y;\theta,s^2)$ over $\theta$ gives $\mathcal N(\bar y;0,\tau^2+s^2)$,
the density of a sum of two independent Gaussians. For the matching in step 3, take
$\Delta_\pi=1/\pi(\hat\theta)$, the width of a flat prior with the same density at
$\hat\theta$.
</details>

**1.** Write $y_i-\theta=(y_i-\bar y)+(\bar y-\theta)$ and expand. The cross term $2(\bar y-\theta)\sum_i(y_i-\bar y)$ vanishes because $\sum_i(y_i-\bar y)=0$. So

$$
p(D\mid\theta)=(2\pi\sigma^2)^{-n/2}e^{-\sum_i(y_i-\bar y)^2/2\sigma^2}\cdot e^{-n(\theta-\bar y)^2/2\sigma^2}.
$$

The first two factors are $p(D\mid\bar y)$, and the last is maximized at $\theta=\bar y$.

**2.** Using the hint,

$$
Z=p(D\mid\bar y)\int\mathcal N(\theta;0,\tau^2)\,e^{-(\theta-\bar y)^2/2s^2}\,d\theta
=p(D\mid\bar y)\,\sqrt{2\pi s^2}\;\mathcal N(\bar y;0,\tau^2+s^2),
$$

and $\sqrt{2\pi s^2}\,\big(2\pi(\tau^2+s^2)\big)^{-1/2}=\sqrt{s^2/(\tau^2+s^2)}$, which gives the stated form.

**3.** Taking $-\ln$:

$$
-\ln Z=-\ln p(D\mid\hat\theta)+\tfrac12\ln\!\Big(1+\frac{\tau^2}{s^2}\Big)+\frac{\bar y^2}{2(\tau^2+s^2)} .
$$

Since $\tau^2/s^2=n\tau^2/\sigma^2$, the middle term is $\tfrac12\ln n+\tfrac12\ln(\tau^2/\sigma^2)+O(1/n)$, and the last term tends to $\bar y^2/2\tau^2$.

Matching: the posterior width is $\Delta_{\text{post}}=\sqrt{2\pi}\,s=\sqrt{2\pi}\,\sigma/\sqrt n$, and $\Delta_\pi=1/\pi(\hat\theta)=\sqrt{2\pi}\,\tau\,e^{\hat\theta^2/2\tau^2}$. Their ratio is $\dfrac{\sigma}{\tau\sqrt n}\,e^{-\hat\theta^2/2\tau^2}$, which is the large-$n$ limit of the Occam factor in step 2. The $\tfrac12\ln n$ is BIC's $\tfrac p2\ln n$ with $p=1$. It comes from the posterior shrinking like $1/\sqrt n$, and the remaining terms are the $O(1)$ that BIC drops.

**4.** The Occam factor behaves like $\sigma/(\tau\sqrt n)\to0$, so $Z\to0$, while $p(D\mid\hat\theta)$ is unchanged. The fit is exactly as good. The model is penalized because a vague prior put almost all its mass on values of $\theta$ that the data ruled out. Evidence comparisons are therefore sensitive to prior widths in a way that likelihood fits are not. This is the first case in Exercise 2.D.i.

### 2.B. Description length: Kolmogorov complexity

Fix a universal prefix machine $U$. Its valid programs form a prefix-free set of
binary strings: no program is a proper prefix of another. The (prefix) Kolmogorov
complexity of a finite string $x$ is

$$
K_U(x)=\min\{\,|p| : U(p)=x\,\}.
$$

Four basic facts, proved in Exercise 2.B.i:

- **Kraft.** If $P$ is prefix-free then $\sum_{p\in P}2^{-|p|}\le1$. Hence
  $\sum_x2^{-K(x)}\le1$, so $2^{-K(x)}$ is a sub-probability over strings: "short
  description" and "high prior probability" are the same thing in two units.
- **Invariance.** For two universal machines $U,V$ there is a constant $c$ with
  $|K_U(x)-K_V(x)|\le c$ for all $x$. Asymptotic statements do not depend on the
  machine, but for a single finite object the constant can exceed the object's
  complexity, so "the Kolmogorov complexity of this trained network" has no
  machine-independent value.
- **Counting.** Fewer than a $2^{-c}$ fraction of $n$-bit strings have $K(x)<n-c$.
  Simplicity is rare, which is why a preference for it carries information.
- **Uncomputability.** $K$ is not computable, though it can be approximated from above.

[Rathmanner and Hutter §5.9](https://arxiv.org/html/1105.5721) (the core reading of §3) gives an informal introduction,
including the reference-machine problem. The proofs below are the standard textbook
arguments.

#### 2.B.i. The four facts

**(a) Kraft.** To a binary string $p=p_1\dots p_k$ assign the half-open interval
$I_p=[0.p_1\dots p_k,\ 0.p_1\dots p_k+2^{-k})\subset[0,1)$, reading $0.p_1\dots p_k$
as a binary fraction. Show that $I_p\cap I_q=\varnothing$ when neither of $p,q$ is a
prefix of the other. Conclude Kraft's inequality, and then $\sum_x2^{-K_U(x)}\le1$.

**(b) Invariance.** Universality of $U$ means that for every prefix machine $V$ there
is a string $s_V$ with $U(s_Vp)=V(p)$ for all $p$. Prove $K_U(x)\le K_V(x)+|s_V|$.
Then, for a fixed string $x^\star$, construct a universal prefix machine $V$ with
$K_V(x^\star)=1$. What does this say about the size of the invariance constant?

**(c) Counting.** Bound the number of programs shorter than $n-c$ bits, and conclude
the counting statement.

**(d) Uncomputability (the Berry argument).** Suppose $K$ were computable. Build a
program $q_m$ that outputs a string of complexity at least $m$, and bound $|q_m|$.
Derive a contradiction. Then explain how $K(x)$ can nevertheless be approximated from
above, and why you can never be sure the approximation has converged.

<details><summary>Hints</summary>

(a) If neither string is a prefix of the other, they first differ at some position $j$.
Look at which half of a dyadic interval of length $2^{-(j-1)}$ each one lies in.

(b) Let $V$ reserve one very short program for $x^\star$ and otherwise defer to $U$.
Check that $V$'s programs are still prefix-free.

(c) How many binary strings have length less than $n-c$? Each program outputs at most
one string.

(d) To write $m$ in a self-delimiting way, write each bit of its binary expansion twice
and end with `01`. That costs about $2\log_2m$ bits.
</details>

**(a)** Suppose $p$ and $q$ first differ at position $j$, say $p_j=0$ and $q_j=1$, and let $a=0.p_1\dots p_{j-1}$ (the common prefix). Every number in $I_p$ lies in $[a,\,a+2^{-j})$ and every number in $I_q$ lies in $[a+2^{-j},\,a+2^{-j+1})$, so the intervals are disjoint. For a prefix-free set $P$, the intervals $I_p$ are therefore pairwise disjoint subintervals of $[0,1)$, and their lengths $2^{-|p|}$ sum to at most 1.

Distinct strings $x$ have distinct shortest programs, all in the prefix-free domain of $U$, so $\sum_x2^{-K_U(x)}\le\sum_{p\in\mathrm{dom}\,U}2^{-|p|}\le1$. (The converse also holds: any lengths satisfying Kraft's inequality are the lengths of some prefix code. That is why code lengths and priors are interchangeable via $L=-\log_2P$.)

**(b)** If $p$ is a shortest $V$-program for $x$, then $s_Vp$ is a $U$-program for $x$ of length $K_V(x)+|s_V|$, so $K_U(x)\le K_V(x)+|s_V|$. Swapping the roles of two universal machines gives the two-sided bound with $c=\max(|s_V|,|s_U|)$.

For the construction, let $V(\texttt{0})=x^\star$ and $V(\texttt{1}p)=U(p)$. Its domain $\{\texttt{0}\}\cup\{\texttt{1}p:p\in\mathrm{dom}\,U\}$ is prefix-free because $\mathrm{dom}\,U$ is. $V$ is universal, since it simulates $U$ at the cost of one bit: $K_V(y)\le K_U(y)+1$ for all $y$. But $K_V(x^\star)=1$, however large $K_U(x^\star)$ is. So the constant relating $U$ and $V$ is at least $K_U(x^\star)-1$. The invariance theorem holds, but it says nothing useful about any single object unless the machine is fixed in advance. Rathmanner and Hutter §5.9 discusses this and proposes restricting to "natural" machines.

**(c)** There are $\sum_{j=0}^{n-c-1}2^j=2^{n-c}-1$ binary strings of length less than $n-c$, and each program outputs at most one string. So fewer than $2^{n-c}$ strings have $K(x)<n-c$, which is a fraction below $2^{-c}$ of the $2^n$ strings of length $n$. For example, fewer than one $n$-bit string in a thousand can be compressed by 10 bits.

**(d)** Let $V$ be the machine that reads a self-delimiting encoding of $m$ and then runs: "for each string $x$, in order of length and then lexicographically, if $K(x)\ge m$ then print $x$ and halt." The loop halts, because fewer than $2^m$ strings have $K<m$. Its output $x_m$ satisfies $K(x_m)\ge m$. By (b), $K_U(x_m)\le 2\log_2m+O(1)$: the $O(1)$ covers $s_V$ and the code that evaluates $K$, and $2\log_2m$ covers the encoding of $m$. So $m\le2\log_2m+O(1)$, which is false for large $m$. Hence $K$ is not computable.

Approximation from above: run all programs in parallel (dovetailing, giving program $i$ a step at stages $i, i+1,\dots$). Let $K_t(x)$ be the length of the shortest program that has halted with output $x$ by stage $t$ (there is always some program of length about $|x|+2\log_2|x|$ that prints $x$ literally). Then $K_t(x)$ is non-increasing in $t$ and equals $K(x)$ from some finite stage on. But you cannot compute that stage: if you could, $K$ would be computable.

### 2.C. MDL: the two-part code

[Source: Grünwald, *A Tutorial Introduction to the Minimum Description Length Principle*, §2.4.2.](https://arxiv.org/html/math/0406077)

Minimum description length fixes a class of codes in advance and scores a hypothesis
$H$ by $L(H)+L(D\mid H)$: the bits to describe $H$, plus the bits to describe the
data using $H$. With $L(D\mid H)=-\log_2p(D\mid H)$ and $L(H)$ from a prefix code,
this is $-\log_2$ of a prior times a likelihood, so it is a MAP criterion with prior
$2^{-L(H)}$. For a continuous parameter you must also decide how precisely to state
it. Exercise 2.C.i shows that the best precision is of order $1/\sqrt n$ and costs
about $\tfrac12\log_2n$ bits per parameter, which recovers BIC at leading order. MDL
is computable by construction, because you chose the code class. Rathmanner and
Hutter §9.2 places MDL relative to Solomonoff induction.

#### 2.C.i. Why precision $1/\sqrt n$?

One parameter $\theta$ ranges over an interval of length $\Delta_\pi$. Near the
maximum-likelihood estimate,
$-\ln p(D\mid\theta)\approx-\ln p(D\mid\hat\theta)+\tfrac{nJ}{2}(\theta-\hat\theta)^2$,
where $J$ is the Fisher information per sample. Assume sender and receiver both know
$n$ and $J$. The code rounds $\hat\theta$ to the nearest point $\theta_q$ of a grid
with spacing $\delta$, sends its index ($\log_2(\Delta_\pi/\delta)$ bits), and then
sends the data using the code $p(\cdot\mid\theta_q)$.

1. Averaging over where $\hat\theta$ falls in its grid cell, $\theta_q-\hat\theta$ is
   uniform on $[-\delta/2,\delta/2]$. Write the expected total length $L(\delta)$ in bits.
2. Minimize $L(\delta)$ over $\delta$.
3. Show $L(\delta^\ast)=-\log_2p(D\mid\hat\theta)+\tfrac12\log_2n+O(1)$, and say what
   the $O(1)$ depends on.
4. Why does the optimal $\delta$ shrink like $1/\sqrt n$?

<details><summary>Hint</summary>

$\mathbb E[(\theta_q-\hat\theta)^2]=\delta^2/12$. Convert nats to bits by dividing by $\ln2$.
</details>

**1.**

$$
L(\delta)=\log_2\frac{\Delta_\pi}{\delta}-\log_2p(D\mid\hat\theta)+\frac{nJ}{2}\cdot\frac{\delta^2}{12}\cdot\frac1{\ln2}.
$$

**2.** $\dfrac{dL}{d\delta}=-\dfrac1{\delta\ln2}+\dfrac{nJ\delta}{12\ln2}=0$ gives $\delta^\ast=\sqrt{12/(nJ)}$.

**3.** $\log_2(\Delta_\pi/\delta^\ast)=\log_2\Delta_\pi+\tfrac12\log_2n+\tfrac12\log_2(J/12)$, and the fit penalty at $\delta^\ast$ is $\dfrac{nJ}{24\ln2}\cdot\dfrac{12}{nJ}=\dfrac1{2\ln2}\approx0.72$ bits. So

$$
L(\delta^\ast)=-\log_2p(D\mid\hat\theta)+\tfrac12\log_2n+\Big[\log_2\Delta_\pi+\tfrac12\log_2\tfrac{J}{12}+\tfrac1{2\ln2}\Big].
$$

The bracket is the $O(1)$. It depends on the parameter's range and on the Fisher information, and both depend on the coordinates you chose for $\theta$ (Exercise 2.D.ii). With $p$ parameters, in coordinates where the Fisher matrix is diagonal, each pays $\tfrac12\log_2n$, for $\tfrac p2\log_2n$ in total: BIC (WB2 §7) in bits.

**4.** $\delta^\ast\approx3.5/\sqrt{nJ}$ is a few posterior standard deviations. Stating $\theta$ more precisely than the data can resolve costs bits and buys no fit, and stating it less precisely costs fit. The $\tfrac12\log_2n$ is the number of bits needed to state $\theta$ to the precision the data can confirm. It is the code-length counterpart of the Occam factor $\Delta_{\text{post}}/\Delta_\pi$ in Exercise 2.A.i.

### 2.D. Where they part

| | penalizes | depends on | computable? |
|---|---|---|---|
| Bayesian Occam | prior volume wasted on data that did not occur | the prior and likelihood | yes, as an integral |
| Kolmogorov | length of the shortest description | the universal machine, up to an additive constant | no |
| MDL two-part | description of hypothesis plus data given hypothesis | the chosen code class and the parameter coordinates | yes |

In a regular parametric model with a smooth prior, evidence and two-part code both
give about $\tfrac p2\log n$ (Exercise 2.A.i and Exercise 2.C.i). They are still
different mechanisms that happen to agree there. *Ideal* MDL, the two-part code with
$L(H)=K(H)$, inherits Kolmogorov's uncomputability. When a paper says "networks are
biased toward simple functions", it is worth asking which of the three it means.

#### 2.D.i. Make them disagree

Give three cases where the volume and description-length notions of simplicity
disagree:

1. a model that is short to describe but heavily Occam-penalized;
2. a hypothesis that is short to describe but that Bayesian model comparison over a
   reasonable model family will not favour;
3. the reverse direction: a model that a parameter-counting code calls expensive but
   that the evidence barely penalizes.

<details><summary>Hint</summary>

For 1, keep one parameter but change the prior (Exercise 2.A.i, step 4). For 2, think
of the digits of $\pi$. For 3, think of a singular model from the SLT day.
</details>

- **1. Short but penalized.** $y=\theta x+\varepsilon$ with $\theta\sim\mathrm{Unif}[-10^{100},10^{100}]$. The model takes one line to state. With $\Delta_{\text{post}}\approx10^{-2}$, the Occam factor is about $10^{-2}/(2\times10^{100})\approx10^{-102}$, a penalty of roughly 340 bits. The volume mechanism cares about the prior's spread, not the model's description.
- **2. Short but not favoured.** The first $10^6$ binary digits of $\pi$. $K$ is small: a short program for $\pi$'s digits plus about 20 bits for the length $10^6$, so at most a few thousand bits on any reasonable machine. A Bayesian comparing i.i.d. or order-$k$ Markov models finds that they all look like fair coins, and the best evidence goes to the fair coin at about $10^6$ bits. Bayesian Occam only ranks the models you put in; Kolmogorov complexity ranks against every program.
- **3. Expensive to a naive code, cheap in volume.** A singular model with many parameters but a small learning coefficient $\lambda$ pays a volume penalty of about $\lambda\ln n$ nats, while a naive two-part code that quantizes every coordinate pays about $\tfrac p2\ln n$. Degenerate directions cost the volume mechanism nothing and cost the naive code bits. (A cleverer code that only states the non-degenerate directions closes some of this gap; the evidence does so automatically.)

#### 2.D.ii. Reparametrize

Let $\varphi=g(\theta)$ be a smooth bijection. Which of these change: the evidence
$Z$; the MAP estimate and the function $f_{\hat\theta}$ it picks; the two-part code
length; $K$ of the fitted function? What does your answer say about "flat minima
generalize" arguments stated in parameter coordinates?

<details><summary>Hint</summary>

The prior density in $\varphi$-coordinates is $\pi_\varphi(\varphi)=\pi(\theta(\varphi))\,|d\theta/d\varphi|$.
The likelihood is a function, not a density, so it picks up no Jacobian. For flatness,
compute $d^2L/d\varphi^2$ at a critical point of $L$.
</details>

- **Evidence: invariant.** $Z_\varphi=\int p(D\mid\theta(\varphi))\,\pi(\theta(\varphi))\,|d\theta/d\varphi|\,d\varphi=\int p(D\mid\theta)\,\pi(\theta)\,d\theta$ by change of variables.
- **MAP: can move.** In $\varphi$-coordinates the MAP maximizes $\ln p(D\mid\theta(\varphi))+\ln\pi(\theta(\varphi))+\ln|d\theta/d\varphi|$, and the extra Jacobian term moves the maximizer unless it is constant. Example with no data: $\theta>0$ with prior $\pi(\theta)=e^{-\theta}$ has its mode at $\theta=0$. In $\varphi=\ln\theta$ the density is $e^{-e^\varphi}e^\varphi$, with mode at $\varphi=0$, i.e. $\theta=1$. So the MAP *function* can change. (The maximum-likelihood estimate does not move, because the likelihood has no Jacobian.)
- **Two-part code: changes under nonlinear $g$.** A uniform grid in $\varphi$ is a non-uniform grid in $\theta$, so it is a different code. In Exercise 2.C.i the $O(1)$ bracket contains $\log_2(\Delta_\pi\sqrt J)$. Under an affine map $\varphi=a\theta+b$, $\Delta_\pi$ scales by $|a|$ and $J$ by $a^{-2}$, so the product is unchanged. Under a nonlinear $g$, $\Delta_\pi$ is a global length, $\int|g'|\,d\theta$, while $J$ rescales by the local factor $g'(\hat\theta)^{-2}$, so the bracket changes in general.
- **$K$ of the fitted function: unchanged** if the fitted function is unchanged, since it is a property of the function. $K$ of the parameter string changes by at most $K(g)+O(1)$ for computable $g$.
- **Flatness.** At a critical point $\theta^\ast$ of a loss $L$, $\dfrac{d^2L}{d\varphi^2}=L''(\theta^\ast)\Big(\dfrac{d\theta}{d\varphi}\Big)^2$. The first-derivative term vanishes because $L'(\theta^\ast)=0$. A reparametrization can therefore make the same minimum, implementing the same function, as sharp or as flat as you like. A flatness measure in raw parameter coordinates is coordinate-dependent, like the two-part code, while the volume (evidence) version is not. This is why the measures in WB2 §9 have to handle reparametrization.

<a id="wb4-section-3"></a>

## 3. Solomonoff

**The result.** Solomonoff's universal prior weights each program by $2^{-|p|}$ and
predicts a binary sequence by mixing everything the programs output. Against any
computable environment (measure) $\mu$, its cumulative log-loss exceeds $\mu$'s own by at most
$K(\mu)\ln 2$ nats (plus a constant), on every sequence and for every length. §3.C
lists what it does not say.

**[A philosophical treatise of universal induction](https://arxiv.org/abs/1105.5721)**, Samuel Rathmanner and Marcus Hutter, 2011 (*Entropy* 13(6), 1076–1136) · *core*

An accessible exposition of Solomonoff induction for non-specialists: Bayes mixtures and their bounds, Kolmogorov complexity, the universal prior and its prediction bounds, computable approximations, and the philosophical objections. About 60 pages. For this notebook: §4.7–4.9 (Bayes mixtures), §5.9 (Kolmogorov complexity), §6.5–7.1 (the universal prior), §8.1 (total bounds), §9 (MDL and computable approximations) and §10.2 (dependence on the reference machine).

### 3.A. The universal prior

Use a *monotone* universal machine $U$. It reads program bits one at a time and
writes output bits it never erases. For a finite binary string $x$, define

$$
M(x)=\sum_{p\ \text{minimal}:\ U(p)\ \text{begins with}\ x} 2^{-|p|},
$$

where "minimal" means no proper prefix of $p$ already outputs something beginning
with $x$. Operationally: feed $U$ fair coin flips as its program; $M(x)$ is the
probability that the output begins with $x$. Short programs get most of the weight.

$M$ is a **semimeasure**, not a measure: $M(\epsilon)\le1$ and
$M(x)\ge M(x0)+M(x1)$, with strict inequality in general, because some programs
print $x$ and then halt, or run forever without printing another bit. $M$ is *lower
semicomputable*: run all programs in parallel and the running sums increase toward
$M(x)$. It is not computable, since you cannot know how much mass is still to come.

### 3.B. Dominance and the regret bound

For a lower-semicomputable $\nu$, $K(\nu)$ denotes the length of a shortest prefix
program that lower-approximates its values, rather than necessarily computing them
exactly. For a computable measure $\mu$ this agrees, up to an additive constant,
with the length of the shortest program that computes $\mu$; the Theorem below uses
the first definition. It is convenient to work with the mixture

$$
\xi(x)=\sum_{\nu} 2^{-K(\nu)}\,\nu(x),
$$

summed over all lower-semicomputable semimeasures $\nu$; the weights sum to at most 1
by Kraft. $\xi$ and $M$ agree up to multiplicative constants (Rathmanner and Hutter
§7–8), so everything below holds for $M$ with $K(\mu)$ replaced by $K(\mu)+O(1)$.
Predict with the conditionals $\xi(a\mid x)=\xi(xa)/\xi(x)$.

**Theorem.** For every computable measure $\mu$, every $n$, and every $x_{1:n}$ with
$\mu(x_{1:n})>0$,

$$
\sum_{t=1}^{n}\Big[-\ln\xi(x_t\mid x_{<t})\Big]-\sum_{t=1}^{n}\Big[-\ln\mu(x_t\mid x_{<t})\Big]
\ \le\ K(\mu)\ln 2 .
$$

This is a bound on the mixture forecaster's cumulative online log-loss, not on any
single hypothesis. Taking expectations under $\mu$, the expected relative entropies
between $\mu$'s and $\xi$'s next-symbol predictions (defined in Exercise 3.B.i(c),
where $\xi$'s conditionals may sum to less than 1) sum, over all time, to at most
$K(\mu)\ln2$, so they tend to zero and the mixture's predictions converge to the truth.
Rathmanner and Hutter §8.1 states the squared-error version and the sharpening to
convergence with $\mu$-probability one. Exercise 3.B.i derives the bound.

This is the Bayes-mixture bound: any Bayesian mixture that gives the truth prior
weight $w_\mu$ has log-loss regret at most $\ln(1/w_\mu)$ (Rathmanner and Hutter
§4.7–4.9). Solomonoff's contribution is the class, which contains every computable
environment, and the weights $2^{-K}$, which make the price "the length of the truth's
shortest description."

#### 3.B.i. Derive the regret bound

Take a countable class of semimeasures $\nu$ with weights $w_\nu>0$,
$\sum_\nu w_\nu\le1$, and let $\xi=\sum_\nu w_\nu\nu$. Suppose the class contains a
measure $\mu$ that generates the data.

**(a)** Show that $\xi(x)\ge w_\mu\,\mu(x)$ for every $x$.

**(b)** Show that $\prod_{t\le n}\xi(x_t\mid x_{<t})=\xi(x_{1:n})/\xi(\epsilon)$, and
conclude that the cumulative log-loss regret of $\xi$ against $\mu$ is at most
$\ln(1/w_\mu)$ on every sequence. Specialize to $w_\nu=2^{-K(\nu)}$.

**(c)** Let $d_t(x_{<t})=\sum_a\mu(a\mid x_{<t})\ln\dfrac{\mu(a\mid x_{<t})}{\xi(a\mid x_{<t})}$.
Show that $d_t\ge0$ even though $\xi(\cdot\mid x_{<t})$ may sum to less than 1. Then
show that $\sum_{t\le n}\mathbb E_\mu[d_t]\le\ln(1/w_\mu)$ for every $n$, and say
what follows as $t\to\infty$.

**(d)** Evaluate the bound for an environment with a 1000-bit description, in nats and
in bits. How does it compare with the cost of simply being told $\mu$?

<details><summary>Hints</summary>

(a) Every term in the sum is non-negative.

(b) Write each conditional as a ratio and let the product telescope. Note $\xi(\epsilon)\le1$
and $\mu(\epsilon)=1$. Then take $-\ln$ of (a).

(c) Use $\ln u\le u-1$ with $u=\xi(a\mid x_{<t})/\mu(a\mid x_{<t})$. For the sum, note
that the conditional expectation of the $t$-th regret term, given $x_{<t}$, is exactly
$d_t(x_{<t})$.
</details>

**(a)** $\xi(x)=\sum_\nu w_\nu\nu(x)$ is a sum of non-negative terms, one of which is $w_\mu\mu(x)$.

**(b)** $\prod_{t\le n}\dfrac{\xi(x_{1:t})}{\xi(x_{<t})}=\dfrac{\xi(x_{1:n})}{\xi(\epsilon)}$, and likewise for $\mu$ with $\mu(\epsilon)=1$. So the regret is

$$
R_n=\sum_{t\le n}\ln\frac{\mu(x_t\mid x_{<t})}{\xi(x_t\mid x_{<t})}=\ln\frac{\mu(x_{1:n})}{\xi(x_{1:n})}+\ln\xi(\epsilon)\ \le\ \ln\frac{\mu(x_{1:n})}{\xi(x_{1:n})}\ \le\ \ln\frac1{w_\mu},
$$

using $\xi(\epsilon)\le1$ and then (a). With $w_\mu=2^{-K(\mu)}$ the bound is $K(\mu)\ln2$. It holds on every sequence and for every $n$: a worst-case guarantee, not an average.

**(c)** Over the symbols $a$ with $\mu_a:=\mu(a\mid x_{<t})>0$, and writing $\xi_a:=\xi(a\mid x_{<t})$,

$$
-d_t=\sum_a\mu_a\ln\frac{\xi_a}{\mu_a}\le\sum_a\mu_a\Big(\frac{\xi_a}{\mu_a}-1\Big)\le\sum_a\xi_a-1\le0 .
$$

So $d_t\ge0$. Given $x_{<t}$, the expected $t$-th regret term is $\sum_a\mu_a\ln(\mu_a/\xi_a)=d_t(x_{<t})$, so by the tower property $\mathbb E_\mu[R_n]=\sum_{t\le n}\mathbb E_\mu[d_t]$. Since $R_n\le\ln(1/w_\mu)$ pointwise, the partial sums of a non-negative series are bounded:

$$
\sum_{t=1}^{\infty}\mathbb E_\mu[d_t]\le K(\mu)\ln2 .
$$

Hence $\mathbb E_\mu[d_t]\to0$, and for any $\varepsilon>0$ at most $K(\mu)\ln2/\varepsilon$ steps can have $\mathbb E_\mu[d_t]>\varepsilon$. For a binary alphabet, Pinsker's inequality gives $(\mu(0\mid\cdot)-\xi(0\mid\cdot))^2\le d_t/2$ when $\xi$'s conditional is normalized, which turns this into a squared-error bound of the kind stated in Rathmanner and Hutter §8.1. The semimeasure case needs an extra step.

**(d)** $1000\ln2\approx693$ nats, which is exactly 1000 bits of total excess code length, however long the sequence. The mixture pays, in total, no more than it would cost to be told the shortest description of $\mu$ up front. This is the MDL reading of Solomonoff induction.

### 3.C. What the theorem is not

- **Not computable.** $M$ is only lower semicomputable. Every practical use is a
  computable surrogate that restricts the class or charges for runtime, for example
  MDL with a fixed code class, resource-bounded priors such as Schmidhuber's speed
  prior, context-tree weighting, and AIXI$tl$. Rathmanner and Hutter §9 surveys such
  approximations.
- **Not a probability measure.** It is a semimeasure, because mass leaks to programs
  that stop printing.
- **Not about functions.** $M$ is a prior over sequences. Supervised learning has
  inputs, so to apply it you must choose an encoding: interleave pairs $(x_i,y_i)$ into
  one sequence, or enumerate a finite input set $x_1,\dots,x_N$ and write a function as
  the string $f(x_1)\cdots f(x_N)$.
- **Not machine-free at finite $n$.** $K_U(\mu)$ depends on $U$ up to an additive
  constant (Exercise 2.B.i(b)). For a finite data budget that constant can dominate.
- **Not approximable by anything simple.** See Legg's reading and Exercise 3.C.i below.

#### 3.C.i. Diagonalize a predictor

**[Is there an elegant universal theory of prediction?](https://arxiv.org/abs/cs/0606070)**, Shane Legg, 2006 · *optional; core for this exercise*

For computable predictors of computable sequences: every predictor is defeated by a sequence of about its own complexity; predictors for all sequences up to complexity $n$ exist but must themselves have complexity about $n$; and beyond some $n$ no such predictor can be proved to work. Short, with each proof about a page; this exercise follows Lemmas 3.2–3.3 and Theorem 4.3.

A *predictor* is a computable map $p$ from finite binary strings to $\{0,1\}$. It
learns $\omega=\omega_1\omega_2\dots$ if there is an $m$ with
$p(\omega_{1:n})=\omega_{n+1}$ for all $n\ge m$ (Legg's Definition 2.4). For an
infinite computable sequence, $K(\omega)$ is the length of a shortest program that
outputs $\omega$ bit by bit on a monotone universal machine (as in §3.A); likewise
$K(p)$ is the length of a shortest program computing the predictor $p$.

**(a)** Given $p$, construct a computable sequence $\omega$ such that
$p(\omega_{1:n})\ne\omega_{n+1}$ for every $n$, and show $K(\omega)\le K(p)+c$ for a
constant $c$ independent of $p$.

**(b)** Deduce that any predictor that learns every computable sequence of complexity
at most $n$ has $K(p)\ge n-c$.

**(c)** Apply (a) to a trained language model with greedy decoding, viewed as a
deterministic next-token predictor. What sequence defeats it, and how complex is that
sequence? What is the analogue if the model is scored by log-loss?

**(d)** Legg's Lemma 3.2 says there is a single simple predictor that learns sequences
of arbitrarily high complexity. Find one, and reconcile it with (b).

<details><summary>Hints</summary>

(a) Let the sequence do the opposite of whatever $p$ predicts.

(b) Apply (a) to such a predictor. The defeating sequence cannot be in the class it learns.

(d) Think of sequences that are complicated only at the start.
</details>

**(a)** Set $\omega_1=1-p(\epsilon)$ and $\omega_{n+1}=1-p(\omega_{1:n})$. A program that computes $\omega$ needs only a shortest program for $p$ plus a fixed wrapper that runs it and flips its outputs, so $K(\omega)\le K(p)+c$. By construction $p$ is wrong at every step, so it never learns $\omega$. This is Legg's Lemma 3.3.

**(b)** Let $p$ learn every computable sequence with $K\le n$. By (a) there is a sequence $\omega'$ with $K(\omega')\le K(p)+c$ that $p$ does not learn, so $\omega'$ is not in that class, i.e. $K(\omega')>n$. Hence $K(p)\ge K(\omega')-c>n-c$. This is Legg's Theorem 4.3.

**(c)** At each step, emit any token other than the model's argmax (say the lowest-ranked token, ties broken by index). The model is wrong at every step. The sequence is computable from the model's weights and decoding code plus a constant-size wrapper, so $K(\text{sequence})\le K(\text{model})+c$. That can be large in absolute terms, but it is no larger than the model. For log-loss, emit the token the model rates least likely: its probability is at most $1/V$ for vocabulary size $V$, so the model pays at least $\ln V$ nats per step. (Legg sketches the same move for probabilistic predictors.) So a claim that a fixed model is a universal predictor has to be distributional ("on the text that actually occurs") or relative to a resource bound, not worst-case.

**(d)** The predictor "always predict 0" learns every sequence of the form $x000\ldots$, and $K(x000\ldots)$ can be as large as you like by choosing $x$ incompressible. There is no conflict with (b). A simple predictor can learn *some* complex sequences, but no simple predictor learns *all* simple ones. What a predictor can learn is not ordered by Kolmogorov complexity.

### 3.D. A bridge to networks

**[Deep learning generalizes because the parameter-function map is biased towards simple functions](https://arxiv.org/abs/1805.08522)**, Guillermo Valle-Pérez, Chico Q. Camargo and Ard A. Louis, ICLR 2019 · *optional*

Argues that the prior over functions induced by random parameters is exponentially biased toward functions of low (proxy) complexity, and builds a PAC-Bayes bound on this prior that is well below chance on binarized MNIST and Fashion-MNIST and near chance on binarized CIFAR-10. The simplicity bound is one-sided and uses a computable proxy for $K$; §7 tests whether SGD samples like the posterior.

<a id="wb4-section-4"></a>

## 4. Program synthesis and the circuit prior

Two accounts that make the simplicity preference concrete: a hypothesis that SGD
searches for short programs, and a theorem that Bayesian fixed-depth MLPs prefer small
circuits.

### 4.A. Readings

**[Deep learning as program synthesis](https://www.lesswrong.com/posts/Dw8mskAvBX37MxvXo/deep-learning-as-program-synthesis-1)**, Zach Furman, January 2026 (LessWrong) · *core*

Argues, as a working hypothesis, that deep learning succeeds because SGD performs a tractable form of program synthesis: a search for simple, compositional algorithms that explain the data, analogous to but far cheaper than Solomonoff induction. Reverse-engineered circuits (grokking, curve detectors, induction heads) are the direct evidence, three "paradoxes" (approximation, generalization, convergence) the indirect evidence, and the representation problem and the search problem are named as what a formal version must solve.

**[A circuit prior in NN-bayes](https://www.lesswrong.com/posts/SqDHeuycNkERurtSc/a-circuit-prior-in-nn-bayes)**, Kaarel Hänni and Dmitry Vaintrob, August 2026 (LessWrong post linking [talk slides](https://kaarelh.github.io/ai/nnbayes-generalization.pdf), "A generalization result for overparametrized neural net Bayesian learning", Iliad 2026)

Shows, in talk slides with proof appendices (a full write-up is forthcoming), that Bayesian learning with an overparametrized fixed-depth MLP, under a strong-prior Gaussian scaling, has a circuit prior: a Boolean target of fixed-depth circuit complexity $C$ is learned from $\mathrm{poly}(d,C)$ samples at any width $N\ge C$, in the sense that the posterior's average probability on the correct label exceeds 99.9%. Slides 10–12 are a warm-up ("Solomonoff function induction") and 15–17 the setting and theorem; slide 34 contrasts this scaling with NNGP, which it describes as needing $\sim d^k$ samples for $k$-sparse parity, and Exercise 4.B.i derives that kernel count alongside a counting learner's $k\log_2d$.

### 4.B. What a prior buys on sparse parity

#### 4.B.i. Sparse parity: counting learner versus kernel learner

Inputs $x$ are uniform on $\{\pm1\}^d$. The target is
$\chi_S(x)=\prod_{i\in S}x_i$ for an unknown set $S$ with $|S|=k$, and labels are
noiseless. Write $N=\binom dk$ for the number of candidate targets.

**(a) A learner with a short-description prior.** Consider the learner that outputs
any $k$-sparse parity consistent with the $n$ training examples. Show that two
distinct parities disagree on exactly half of all inputs. Deduce that, with
probability at least $1-\delta$, the only consistent parity is the target once
$n\ge\log_2N+\log_2(1/\delta)$, and that $\log_2N\le k\log_2d$.

**(b) A fixed-kernel learner.** Kernel ridge regression, minimum-norm kernel
interpolation and the Gaussian-process posterior mean all output a function of the form
$\hat f=\sum_{i=1}^n\alpha_iK(x_i,\cdot)$, so $\hat f$ lies in
$V=\mathrm{span}\{K(x_i,\cdot)\}$. Note that $V$ depends on the inputs but not on the
labels. Working in $L^2$ of the uniform distribution, show that
$\sum_{|S|=k}\lVert P_V\chi_S\rVert^2\le n$, where $P_V$ is orthogonal projection onto
$V$. Deduce that, averaged over a uniformly random target $S$,
$\mathbb E_S\lVert\hat f-\chi_S\rVert^2\ge1-n/N$.

**(c)** Put numbers in for $d=100$, $k=5$. Then say what each result does and does not
tell you about networks trained by SGD.

<details><summary>Hints</summary>

(a) $\chi_S\chi_T=\chi_{S\triangle T}$, and a non-constant parity has mean zero. Then use
a union bound over the $N-1$ wrong parities.

(b) The parities $\chi_S$ are orthonormal in $L^2$ of the uniform distribution. Take an
orthonormal basis $e_1,\dots,e_r$ of $V$ ($r\le n$), expand
$\lVert P_V\chi_S\rVert^2=\sum_j\langle\chi_S,e_j\rangle^2$, swap the sums, and use
Bessel's inequality. Then $\lVert\hat f-\chi_S\rVert^2\ge\lVert\chi_S-P_V\chi_S\rVert^2=1-\lVert P_V\chi_S\rVert^2$.
</details>

**(a)** For $S\ne T$, $\chi_S(x)\chi_T(x)=\chi_{S\triangle T}(x)$ with $S\triangle T\ne\varnothing$, and $\mathbb E_x[\chi_{S\triangle T}(x)]=\prod_{i\in S\triangle T}\mathbb E[x_i]=0$. So $\chi_S$ and $\chi_T$ agree on exactly half the inputs, and a fixed wrong parity is consistent with $n$ i.i.d. examples with probability $2^{-n}$. By a union bound over the $N-1$ wrong parities, the probability that any of them is consistent is at most $N2^{-n}$, which is at most $\delta$ once $n\ge\log_2N+\log_2(1/\delta)$. Finally $N=\binom dk\le d^k$, so $\log_2N\le k\log_2d$.

$\log_2N$ is the number of bits needed to name $S$. This is the Occam bound in miniature: samples scale with description length. It is also the shape of the "Solomonoff function induction" warm-up in the circuit-prior slides (slides 10–12), where a target with an $O(C)$-bit description is learned from $O(C)$ samples.

**(b)** Let $e_1,\dots,e_r$ be an orthonormal basis of $V$, with $r\le n$. Then

$$
\sum_{|S|=k}\lVert P_V\chi_S\rVert^2=\sum_{|S|=k}\sum_{j=1}^r\langle\chi_S,e_j\rangle^2=\sum_{j=1}^r\sum_{|S|=k}\langle e_j,\chi_S\rangle^2\le\sum_{j=1}^r\lVert e_j\rVert^2=r\le n,
$$

where the inequality is Bessel's inequality for the orthonormal set $\{\chi_S\}$. Because $\hat f\in V$ and $\lVert\chi_S\rVert=1$,

$$
\lVert\hat f-\chi_S\rVert^2\ \ge\ \min_{g\in V}\lVert g-\chi_S\rVert^2=1-\lVert P_V\chi_S\rVert^2 .
$$

Averaging over the $N$ equally likely targets gives $\mathbb E_S\lVert\hat f-\chi_S\rVert^2\ge1-n/N$. The trivial predictor $\hat f=0$ has error exactly 1, so to do substantially better than nothing a fixed-kernel method needs $n$ of order $N=\binom dk\approx d^k/k!$. The argument uses nothing about the kernel. It also covers a single draw from a Gaussian-process posterior: such a draw is the posterior mean plus noise whose distribution depends only on the inputs, and adding independent zero-mean noise can only increase the expected squared error. Squared error is not 0–1 error, so this is a statement about regression-style fits.

**(c)** For $d=100$, $k=5$: $N=\binom{100}5=75{,}287{,}520$ and $\log_2N\approx26.2$, so (a) needs about 33 samples for $\delta=0.01$, while (b) needs on the order of $7.5\times10^7$.

What this does tell you: (a) is the cost for a learner that pays about $\log_2N$ bits to name the target, as a description-length prior does (the program-length row of the §6 table; the circuit-prior theorem gives the cruder but still polynomial $\mathrm{poly}(d,C)$). (b) is the cost under any fixed-kernel prior such as NNGP (the kernel-norm row). This is the $d^k$ in slide 34, made precise for kernel methods.

What it does not tell you: (a) is information-theoretic. The learner as described enumerates $N$ candidates. (Noiseless parity can in fact be learned from about $d$ samples in polynomial time by Gaussian elimination over $\mathrm{GF}(2)$, but SGD on a network is not Gaussian elimination.) Statistical-query-style lower bounds suggest that gradient methods need time growing like $d^{\Omega(k)}$ on this task, although the information-theoretic sample cost is small. So the gap between (a) and what SGD does is a fact about *search* (the search problem Furman names), not about the prior.

<a id="wb4-section-5"></a>

## 5. Hierarchical representations

A different answer puts the structure in the target rather than in the learner's
prior.

### 5.A. Compositional targets

A depth-matched network approximates a binary tree of 2-variable functions with
$\sim(d-1)\varepsilon^{-2/r}$ units versus $\varepsilon^{-d/r}$ for a shallow one (an
existence result: Furman's "paradox of approximation").
[Source and smoothness assumptions: Poggio et al., Theorems 1–2.](https://arxiv.org/html/1611.00740)

### 5.B. A solvable learning model: the Random Hierarchy Model

**[How deep neural networks learn compositional data: the Random Hierarchy Model](https://arxiv.org/abs/2307.02129)**, Francesco Cagnetta, Leonardo Petrini, Umberto M. Tomasini, Alessandro Favero and Matthieu Wyart, 2024 (*Phys. Rev. X* 14, 031001) · *core*

On a synthetic family of hierarchical tasks, deep networks trained by gradient descent need a number of samples polynomial in the input dimension, about $n_c m^L$, while shallow networks need a number exponential in it. That sample size coincides with the onset of synonym-invariant representations and with the point where low-level feature–label correlations become detectable; removing those correlations makes the task exponentially hard even for deep networks. §I defines the model, §II gives the shallow-versus-deep scaling, and §III the correlation argument, ending with the uncorrelated control.

The model ingredients the exercise below uses:

- **Vocabulary and patches.** Each level has $v$ symbols. Each symbol at level $\ell$
  is rewritten as a patch of $s$ symbols at level $\ell-1$.
- **Synonyms.** Each symbol has $m$ equivalent rewrites ("synonyms"), chosen at random
  once and then fixed, with $m\le v^{s-1}$ so that rewrites are unambiguous.
- **Generation.** A class label (one of $n_c$) is rewritten $L$ times, which gives an
  input of length $d=s^L$.

Deep CNNs trained by gradient descent need about $P^*\simeq n_c\,m^L$ samples.

#### 5.B.i. Where does the learner's simplicity enter?

Estimate how many samples an idealized learner with a program-length prior would need
on the Random Hierarchy Model, and compare it with $P^*\simeq n_cm^L$ as $L$ grows.
What does the comparison suggest about what explains the observed $P^*$?

<details><summary>Hint</summary>

Count the bits needed to write down all the rules, level by level. In the warm-up of
the circuit-prior slides (§4.A, slides 10–12), a Bayesian learner over programs needs
a number of samples linear in the target's description length; also, each label
carries at most $\log_2n_c$ bits.
</details>

- **Description length of the rules.** Each of $L$ levels has about $v$ symbols, each with $m$ rewrites of $s$ symbols at $\log_2v$ bits per symbol: about $L\,v\,m\,s\log_2v$ bits.
- **Samples for the idealized learner.** Each label gives at most $\log_2n_c$ bits, so roughly $L\,v\,m\,s\log_2v/\log_2n_c$ samples, up to constants. This is *linear* in $L$. The observed $P^*\simeq n_cm^L$ is *exponential* in $L$, though polynomial in $d=s^L$.
- **Example.** $v=n_c=8$, $m=4$, $s=2$ (which satisfies $m\le v^{s-1}$), $L=3$: about $3\cdot8\cdot4\cdot2\cdot3/3=192$ versus $8\cdot4^3=512$, the same order. At $L=6$: about 384 versus 32,768.
- **Interpretation.** The idealized length-prior learner would be far more sample-efficient, but it is intractable. The deep network's actual cost tracks the detectability of correlations, which suggests that the observed $P^*$ reflects the search mechanism and the data's statistics rather than a program-length prior.
- **The uncorrelated variant** sharpens the point. The rules are equally short, so the idealized learner is unaffected, while deep networks become exponential.
- **Caveats.** This is a back-of-envelope estimate, and the paper does not test a length-prior learner. Unlabelled inputs also carry information about the rules here, since they are generated by the grammar, so the label-only count overstates what an idealized learner needs.

<a id="wb4-section-6"></a>

## 6. The accounts side by side

| prior over functions | where in this notebook | sample cost, $k$-sparse parity on $\{\pm1\}^d$ | correlated vs uncorrelated hierarchy (§5.B) |
|---|---|---|---|
| program length, $2^{-\ell(f)}$ | §2.B, §3; Furman (§4.A) | $\approx k\log_2d$, information-theoretic (Exercise 4.B.i(a)) | rules equally short, so rated equally hard |
| circuit size at fixed depth, $C_L(f)$ | Hänni–Vaintrob (§4.A) | $\mathrm{poly}(d,C)$, for Bayesian learning at any width $N\ge C$ | not addressed here |
| kernel norm, $e^{-\lVert f\rVert_{\mathcal H}^2/2}$ (NNGP, lazy/NTK; kernel-ridge learning curves in WB5 §7) | Exercise 4.B.i(b) | $\gtrsim\binom dk$ for any fixed kernel | not addressed here |
| *for comparison:* deep networks trained by gradient methods | Exercise 4.B.i(c), §5.B | time believed to grow like $d^{\Omega(k)}$ (statistical-query-style bounds) | polynomial ($\approx n_cm^L$) vs exponential in $d$ |

SLT describes posterior concentration in parameter space (§2.A's volume, done correctly
for singular models), not a fourth prior; see the SLT day.

## What carries forward

- **WB5 §2–3.** Different architectures learning similar representations is the "paradox of convergence" that program synthesis claims to explain; WB5 §2 is about how such similarity is measured and §3 about why it might arise. Linear and superposed feature representations are the subject of WB5 §4–5.
- **WB2 §7–8.** Evidence, Solomonoff and the circuit-prior theorem are statements about Bayesian predictors. Transferring these guarantees to a particular trained network requires an argument linking the theorem's predictor to that network. Posterior sampling is one possible link, and WB2 §8 tests whether SGD's stationary covariance on a quadratic basin matches a tempered posterior's; PAC-Bayes itself does not require the training algorithm to sample a Bayesian posterior.
- **WB3 §3.** Cross-entropy is code length, so a language model is a compressor. That is MDL in practice, and §2–3 here are its theory.
- **WB6.** One of its readings asks whether circuits are the right unit of explanation, which both program synthesis and the circuit prior assume.

## Further study

- Vaintrob and Hänni, ["Width-robust learnability in mean-field Bayesian neural networks"](https://arxiv.org/abs/2607.05735) (2026), for the mean-field (infinite-width) counterpart of the circuit-prior result.